# Customer Churn Classification — Improved Evaluation

This notebook compares Logistic Regression, Decision Tree, Random Forest, Linear SVM, and XGBoost using a stratified split from `training.csv`. It selects a model using validation F1, tunes a probability threshold on validation data where supported, and reports the final confusion matrix on an untouched holdout set.

**Data-quality note:** The supplied `training.csv` and `testing.csv` contain many overlapping `CustomerID` values with conflicting churn labels. They do not appear to be a conventional train/test split of the same records. For a fair primary evaluation, this notebook creates training, validation, and holdout subsets from `training.csv` and inspects `testing.csv` separately.

Near-perfect scores should be verified on genuinely independent, consistently generated data before being presented as real-world performance.

In [ ]:
# If needed, uncomment and run:
# %pip install pandas numpy matplotlib seaborn scikit-learn xgboost

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay
)

RANDOM_STATE = 42


## 1. Load and inspect the files

In [ ]:
train = pd.read_csv("training.csv")
external_test = pd.read_csv("testing.csv")

train = train.dropna(how="all").dropna(subset=["Churn"]).copy()
external_test = external_test.dropna(how="all").dropna(subset=["Churn"]).copy()

train["Churn"] = train["Churn"].astype(int)
external_test["Churn"] = external_test["Churn"].astype(int)

print("training.csv:", train.shape)
print("testing.csv:", external_test.shape)
display(train.head())

print("\nTraining target distribution:")
display(train["Churn"].value_counts().rename_axis("Churn").to_frame("Count"))
display((train["Churn"].value_counts(normalize=True) * 100).round(2).rename("Percent"))


In [ ]:
# Check whether the files really represent independent records.
if "CustomerID" in train.columns and "CustomerID" in external_test.columns:
    overlap = set(train["CustomerID"].dropna()) & set(external_test["CustomerID"].dropna())
    print("CustomerID overlap count:", len(overlap))
    if overlap:
        left = train[["CustomerID", "Churn"]].drop_duplicates("CustomerID")
        right = external_test[["CustomerID", "Churn"]].drop_duplicates("CustomerID")
        joined = left.merge(right, on="CustomerID", suffixes=("_train", "_test"))
        disagreement = (joined["Churn_train"] != joined["Churn_test"]).mean()
        print(f"Overlapping IDs with conflicting labels: {disagreement:.1%}")
        print("Because of this overlap/conflict, testing.csv is not used for model selection.")


## 2. Create training, validation, and final holdout splits

- 70% of `training.csv`: fit models
- 15%: choose the model and tune the threshold
- 15%: final holdout evaluation

The final holdout is not used to choose the model or threshold. `CustomerID` is excluded because it is an identifier rather than a meaningful behavior feature.

In [ ]:
target = "Churn"
drop_columns = [target]
if "CustomerID" in train.columns:
    drop_columns.append("CustomerID")

X_all = train.drop(columns=drop_columns)
y_all = train[target].astype(int)

X_dev, X_holdout, y_dev, y_holdout = train_test_split(
    X_all, y_all, test_size=0.15, random_state=RANDOM_STATE, stratify=y_all
)

validation_fraction_of_dev = 0.15 / 0.85
X_fit, X_val, y_fit, y_val = train_test_split(
    X_dev, y_dev, test_size=validation_fraction_of_dev,
    random_state=RANDOM_STATE, stratify=y_dev
)

print("Fit:", X_fit.shape, "Validation:", X_val.shape, "Final holdout:", X_holdout.shape)
print("\nChurn rate:")
print("Fit:", round(y_fit.mean(), 4))
print("Validation:", round(y_val.mean(), 4))
print("Holdout:", round(y_holdout.mean(), 4))


## 3. Build preprocessing pipelines

In [ ]:
numeric_features = X_fit.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_fit.select_dtypes(exclude=["number"]).columns.tolist()

def make_preprocessor(scale_numeric=False):
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))

    numeric_pipeline = Pipeline(numeric_steps)
    categorical_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    return ColumnTransformer(transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features)
    ])

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)


## 4. Train and compare five classifiers

These are starting configurations, not guaranteed optimal hyperparameters. `LinearSVC` is used instead of an RBF-kernel SVM because an RBF SVM is usually impractical on hundreds of thousands of rows. Each pipeline fits preprocessing only on the fitting split.

In [ ]:
model_configs = {
    "Logistic Regression": (
        LogisticRegression(max_iter=1500, C=1.0, random_state=RANDOM_STATE), True
    ),
    "Decision Tree": (
        DecisionTreeClassifier(max_depth=18, min_samples_leaf=5, random_state=RANDOM_STATE), False
    ),
    "Random Forest": (
        RandomForestClassifier(
            n_estimators=180, min_samples_leaf=2, max_features="sqrt",
            n_jobs=-1, random_state=RANDOM_STATE
        ), False
    ),
    "Linear SVM": (
        LinearSVC(C=1.0, max_iter=5000, random_state=RANDOM_STATE), True
    ),
    "XGBoost": (
        XGBClassifier(
            n_estimators=350, max_depth=6, learning_rate=0.05,
            min_child_weight=2, subsample=0.9, colsample_bytree=0.9,
            reg_alpha=0.0, reg_lambda=3.0,
            objective="binary:logistic", eval_metric="logloss",
            tree_method="hist", n_jobs=-1, random_state=RANDOM_STATE
        ), False
    )
}

fitted_models = {}
comparison_rows = []

for model_name, (estimator, scale_numeric) in model_configs.items():
    print(f"Training {model_name}...")
    pipeline = Pipeline(steps=[
        ("preprocessor", make_preprocessor(scale_numeric=scale_numeric)),
        ("classifier", estimator)
    ])
    pipeline.fit(X_fit, y_fit)
    pred = pipeline.predict(X_val)

    if hasattr(pipeline, "predict_proba"):
        score = pipeline.predict_proba(X_val)[:, 1]
    else:
        score = pipeline.decision_function(X_val)

    fitted_models[model_name] = pipeline
    comparison_rows.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_val, pred),
        "Precision": precision_score(y_val, pred, zero_division=0),
        "Recall": recall_score(y_val, pred, zero_division=0),
        "F1 Score": f1_score(y_val, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_val, score)
    })

comparison_df = pd.DataFrame(comparison_rows).sort_values(
    "F1 Score", ascending=False
).reset_index(drop=True)

display(comparison_df.style.format({
    "Accuracy": "{:.4f}", "Precision": "{:.4f}",
    "Recall": "{:.4f}", "F1 Score": "{:.4f}", "ROC-AUC": "{:.4f}"
}))


## 5. Select a model and tune its threshold on validation data

The model is selected by validation F1. If the selected model provides probabilities, test thresholds on the validation split and choose the one with the best F1 (using accuracy as a tie-breaker). The chosen threshold is fixed before the final holdout evaluation.

In [ ]:
best_model_name = comparison_df.iloc[0]["Model"]
best_model = fitted_models[best_model_name]
best_threshold = 0.5
threshold_table = None

print("Best model by validation F1:", best_model_name)

if hasattr(best_model, "predict_proba"):
    val_scores = best_model.predict_proba(X_val)[:, 1]
    threshold_rows = []

    for threshold in np.arange(0.10, 0.91, 0.01):
        pred = (val_scores >= threshold).astype(int)
        threshold_rows.append({
            "Threshold": threshold,
            "Accuracy": accuracy_score(y_val, pred),
            "Precision": precision_score(y_val, pred, zero_division=0),
            "Recall": recall_score(y_val, pred, zero_division=0),
            "F1 Score": f1_score(y_val, pred, zero_division=0)
        })

    threshold_table = pd.DataFrame(threshold_rows)
    best_row = threshold_table.sort_values(
        ["F1 Score", "Accuracy"], ascending=False
    ).iloc[0]
    best_threshold = float(best_row["Threshold"])

    display(threshold_table.sort_values(
        ["F1 Score", "Accuracy"], ascending=False
    ).head(10).style.format({
        "Threshold": "{:.2f}", "Accuracy": "{:.4f}",
        "Precision": "{:.4f}", "Recall": "{:.4f}", "F1 Score": "{:.4f}"
    }))

print("Selected threshold:", round(best_threshold, 2))


## 6. Final holdout metrics and confusion matrix

In [ ]:
# Keep the holdout untouched until this point.
if hasattr(best_model, "predict_proba"):
    holdout_scores = best_model.predict_proba(X_holdout)[:, 1]
    holdout_pred = (holdout_scores >= best_threshold).astype(int)
else:
    holdout_scores = best_model.decision_function(X_holdout)
    holdout_pred = best_model.predict(X_holdout)

holdout_metrics = {
    "Accuracy": accuracy_score(y_holdout, holdout_pred),
    "Precision": precision_score(y_holdout, holdout_pred, zero_division=0),
    "Recall": recall_score(y_holdout, holdout_pred, zero_division=0),
    "F1 Score": f1_score(y_holdout, holdout_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_holdout, holdout_scores)
}

print(f"Final holdout results — {best_model_name}")
for metric, value in holdout_metrics.items():
    print(f"{metric:10s}: {value:.4f}")

print("\nClassification report:")
print(classification_report(
    y_holdout, holdout_pred,
    target_names=["No Churn", "Churn"], zero_division=0
))

cm = confusion_matrix(y_holdout, holdout_pred, labels=[0, 1])
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["No Churn (0)", "Churn (1)"]
)
disp.plot(cmap="Blues", values_format="d")
plt.title(f"Final Holdout Confusion Matrix — {best_model_name}")
plt.tight_layout()
plt.show()


## 7. Compare validation metrics visually

In [ ]:
metrics_to_plot = ["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"]
plot_df = comparison_df.melt(
    id_vars="Model", value_vars=metrics_to_plot,
    var_name="Metric", value_name="Score"
)

plt.figure(figsize=(13, 6))
sns.barplot(data=plot_df, x="Model", y="Score", hue="Metric")
plt.title("Validation Performance by Model")
plt.ylabel("Score")
plt.ylim(0, 1.05)
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


## 8. Save comparison results and holdout predictions

In [ ]:
comparison_df.to_csv("churn_model_comparison_improved.csv", index=False)

holdout_output = pd.DataFrame({
    "Actual_Churn": y_holdout.to_numpy(),
    "Predicted_Churn": holdout_pred,
    "Score": holdout_scores
})
holdout_output.to_csv("churn_holdout_predictions.csv", index=False)

print("Saved churn_model_comparison_improved.csv")
print("Saved churn_holdout_predictions.csv")


## How to interpret the results

- **Accuracy:** fraction of all predictions that are correct.
- **Precision:** fraction of predicted churners who actually churned.
- **Recall:** fraction of actual churners the model detected.
- **F1 score:** harmonic mean of precision and recall.
- **Confusion matrix:** rows are actual classes and columns are predicted classes. Top-left = true negatives, top-right = false positives, bottom-left = false negatives, bottom-right = true positives.

Do not assume the best validation score guarantees the same score on new data. The supplied files show overlapping customer IDs with inconsistent labels, so validate the project on a clean, independently sampled dataset before claiming real-world performance. A very high score on a random split can also mean the labels are strongly encoded by the features or the dataset is synthetic.